# Team Game - 03_clean_team_game

This notebook merges the team-game datasets by `GAME_ID` and `TEAM_ID`.

Source folder: `02_data/01_raw/2025_26/03_team_game`

Scope:
- Load `games_base` and merge `boxscore_advanced_v3`, `boxscore_four_factors_v3`, `boxscore_misc_v3`
- Standardize key columns
- Report missing values (null cells) overall and by row/column distribution
- Export merged parquet to `02_data/02_cleaned/2025_26/03_team_game/team_game.parquet`


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

data_root = project_root / "02_data" / "01_raw" / "2025_26" / "03_team_game"

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [2]:
endpoints = ["games_base", "boxscore_advanced_v3", "boxscore_four_factors_v3", "boxscore_misc_v3"]

files_summary = []
for endpoint in endpoints:
    endpoint_dir = data_root / endpoint
    files = sorted(endpoint_dir.glob("*.parquet"))
    files_summary.append({
        "endpoint": endpoint,
        "files": len(files),
        "total_mb": round(sum(f.stat().st_size for f in files) / 1e6, 3) if files else 0,
    })

pd.DataFrame(files_summary)


,endpoint,files,total_mb
0,games_base,895,30.403
1,boxscore_advanced_v3,898,19.155
2,boxscore_four_factors_v3,898,11.007
3,boxscore_misc_v3,898,12.554


In [3]:
def load_endpoint(endpoint):
    endpoint_dir = data_root / endpoint
    files = sorted(endpoint_dir.glob("*.parquet"))
    dfs = [pd.read_parquet(f) for f in files]
    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    return df

def standardize_keys(df):
    if df.empty:
        return df

    if "GAME_ID" not in df.columns and "gameId" in df.columns:
        df["GAME_ID"] = df["gameId"]
    if "TEAM_ID" not in df.columns and "teamId" in df.columns:
        df["TEAM_ID"] = df["teamId"]

    if "GAME_ID" in df.columns:
        df["GAME_ID"] = df["GAME_ID"].astype(str)
    if "TEAM_ID" in df.columns:
        df["TEAM_ID"] = pd.to_numeric(df["TEAM_ID"], errors="coerce").astype("Int64")

    drop_cols = [c for c in ["gameId", "teamId"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    return df

def dedupe_on_keys(df, name, keys=("GAME_ID", "TEAM_ID")):
    if df.empty:
        return df

    if not all(k in df.columns for k in keys):
        print(f"[{name}] Missing key columns: {keys}")
        return df

    dup_mask = df.duplicated(list(keys), keep=False)
    dup_count = int(dup_mask.sum())
    if dup_count:
        print(f"[{name}] Duplicate rows on keys: {dup_count}")
        df = df.drop_duplicates(list(keys), keep="first")
    return df

data = {}
for endpoint in endpoints:
    df = load_endpoint(endpoint)
    df = standardize_keys(df)
    data[endpoint] = df
    print(f"{endpoint}: shape={df.shape}")


games_base: shape=(1790, 60)
boxscore_advanced_v3: shape=(1796, 33)
boxscore_four_factors_v3: shape=(1796, 18)
boxscore_misc_v3: shape=(1796, 22)


In [4]:
base = data["games_base"].copy()
base = dedupe_on_keys(base, "games_base")

def merge_extra(base_df, extra_df, name):
    if extra_df.empty:
        print(f"Skipping {name}: empty dataset")
        return base_df

    extra_df = dedupe_on_keys(extra_df, name)
    keys = ["GAME_ID", "TEAM_ID"]

    overlap = [c for c in extra_df.columns if c in base_df.columns and c not in keys]
    if overlap:
        print(f"{name}: dropping {len(overlap)} overlapping columns")
        extra_df = extra_df.drop(columns=overlap)

    merged = base_df.merge(extra_df, on=keys, how="left")
    print(f"After merge {name}: {merged.shape}")
    return merged

merged = base
merged = merge_extra(merged, data["boxscore_advanced_v3"], "boxscore_advanced_v3")
merged = merge_extra(merged, data["boxscore_four_factors_v3"], "boxscore_four_factors_v3")
merged = merge_extra(merged, data["boxscore_misc_v3"], "boxscore_misc_v3")

merged.shape


boxscore_advanced_v3: dropping 3 overlapping columns
After merge boxscore_advanced_v3: (1790, 88)
boxscore_four_factors_v3: dropping 10 overlapping columns
After merge boxscore_four_factors_v3: (1790, 94)
boxscore_misc_v3: dropping 8 overlapping columns
After merge boxscore_misc_v3: (1790, 106)


(1790, 106)

In [5]:
rows, cols = merged.shape
total_cells = rows * cols
null_cells = int(merged.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,1790,106,189740,0,0.0


In [6]:
if merged.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = merged.isna().sum()
    col_null_pct = merged.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
SEASON_YEAR,0,0.0
effectiveFieldGoalPercentage,0,0.0
estimatedTeamTurnoverPercentage,0,0.0
reboundPercentage,0,0.0
defensiveReboundPercentage,0,0.0
offensiveReboundPercentage,0,0.0
assistRatio,0,0.0
assistToTurnover,0,0.0
assistPercentage,0,0.0
netRating,0,0.0


In [7]:
if merged.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = merged.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",1790,100.0
"(0.01, 0.05]",0,0.0
"(0.05, 0.1]",0,0.0
"(0.1, 0.25]",0,0.0
"(0.25, 0.5]",0,0.0
"(0.5, 0.75]",0,0.0
"(0.75, 1.0]",0,0.0


In [8]:
if merged.empty:
    col_dist = pd.DataFrame()
else:
    col_null_pct = merged.isna().mean()
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    col_bins = pd.cut(col_null_pct, bins=bins, include_lowest=True)
    col_counts = col_bins.value_counts().sort_index()
    col_dist = pd.DataFrame({
        "columns": col_counts,
        "pct_columns": (col_counts / len(col_null_pct) * 100).round(3),
    })

col_dist


,columns,pct_columns
"(-0.001, 0.01]",106,100.0
"(0.01, 0.05]",0,0.0
"(0.05, 0.1]",0,0.0
"(0.1, 0.25]",0,0.0
"(0.25, 0.5]",0,0.0
"(0.5, 0.75]",0,0.0
"(0.75, 1.0]",0,0.0


In [9]:
out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "03_team_game"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "team_game.parquet"

merged.to_parquet(out_path, index=False)
print(f"Saved to: {out_path}")


Saved to: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/03_team_game/team_game.parquet
